# 基于 40% 蛋白序列相似性的 cluster split

本 notebook 生成两个 Interformer-compatible 的划分版本，并用散点图观察 train / valid / test 三个类别的数据范围。

1. `interformer_assigned`：只对 `split/` 中原 Interformer train/valid/test 的并集重新分割。
2. `all_raw`：对本地 raw PDBbind index 中的全部 rows 重新分割。

两者都保持 Interformer 训练接口需要的文件名：

- `timesplit_no_lig_overlap_train`
- `timesplit_no_lig_overlap_val`
- `timesplit_test`
- `coresetlist`
- `diff_test+core`
- `posebusters_pdb_ccd_ids.txt`
- `timesplit_test_no_rec_overlap`
- `timesplit_test_sanitizable`

核心约束：如果两条蛋白链在 MMseqs2 中满足 `identity >= 40%` 且 `coverage >= 80%`，它们所在的 PDB complex 会被放入同一个 sequence component，整个 component 只能进入一个 split。


In [ ]:
from pathlib import Path
import json
import os

import pandas as pd
from IPython.display import HTML, display

# Matplotlib 在当前沙箱中不能写 ~/.matplotlib，因此把缓存目录放到项目内。
MPLCONFIGDIR = Path('data/processed/matplotlib_cache')
XDG_CACHE_HOME = Path('data/processed/font_cache')
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
XDG_CACHE_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIGDIR.resolve()))
os.environ.setdefault('XDG_CACHE_HOME', str(XDG_CACHE_HOME.resolve()))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from matplotlib import font_manager

# 配置中文字体。macOS 上优先使用覆盖面较好的 Arial Unicode MS，
# 找不到时回退到 Hiragino Sans GB / STHeiti / Songti SC。
CJK_FONT_CANDIDATES = [
    Path('/Library/Fonts/Arial Unicode.ttf'),
    Path('/System/Library/Fonts/Supplemental/Arial Unicode.ttf'),
    Path('/System/Library/Fonts/Hiragino Sans GB.ttc'),
    Path('/System/Library/Fonts/STHeiti Medium.ttc'),
    Path('/System/Library/Fonts/Supplemental/Songti.ttc'),
]
CJK_FONT_NAME = None
for font_path in CJK_FONT_CANDIDATES:
    if font_path.exists():
        font_manager.fontManager.addfont(str(font_path))
        CJK_FONT_NAME = font_manager.FontProperties(fname=str(font_path)).get_name()
        break
if CJK_FONT_NAME:
    plt.rcParams['font.family'] = [CJK_FONT_NAME, 'DejaVu Sans']
else:
    plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Hiragino Sans GB', 'STHeiti', 'Songti SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


from scripts.create_sequence_cluster_split import create_sequence_cluster_split

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 120)
plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

MPLCONFIGDIR


In [ ]:
# =============================
# 参数入口：两个版本的分割配置
# =============================
# Interformer 原始 split 目录。目录内应包含 timesplit_no_lig_overlap_train / val 和 timesplit_test。
INTERFORMER_SPLIT_DIR = Path('split')

# 版本 1：只对 Interformer 原 train/valid/test 并集重新分割。
INTERFORMER_UNION_OUTPUT_SPLIT_DIR = Path('split_sequence_cluster_interformer_union')
INTERFORMER_UNION_OUTPUT_DIR = Path('data/processed/sequence_cluster_split_interformer_union')

# 版本 2：对全部 raw PDBbind rows 重新分割，比例沿用 Interformer train/valid/test 比例。
ALL_RAW_OUTPUT_SPLIT_DIR = Path('split_sequence_cluster_all_raw')
ALL_RAW_OUTPUT_DIR = Path('data/processed/sequence_cluster_split_all_raw')

# 序列相似性阈值。跨 split 中不允许出现 identity >= 40% 且 coverage >= 80% 的蛋白链命中。
MIN_SEQ_ID = 0.4
COVERAGE = 0.8
MIN_CHAIN_LENGTH = 30

# 多 seed 用于在 component 级别寻找更接近目标比例和标签分布的划分。
OPTIMIZATION_SEEDS = 256

# 设为 None 表示对当前 universe 重新生成 MMseqs all-vs-all 命中表。
# 不建议复用其它 universe 的旧 m8 文件，否则可能与后续独立检测口径不一致。
ALL_VS_ALL_M8 = None

INTERFORMER_SPLIT_DIR, INTERFORMER_UNION_OUTPUT_SPLIT_DIR, ALL_RAW_OUTPUT_SPLIT_DIR


In [ ]:
# =============================
# Matplotlib 可视化工具
# =============================
def plot_split_range(df, title):
    """用 release_year-pAffinity 散点图查看 train/valid/test 三类数据范围。"""
    plot_df = df[df['split'].isin(['train', 'valid', 'test'])].copy()
    plot_df['release_year_numeric'] = pd.to_numeric(plot_df['release_year'], errors='coerce')
    plot_df['pAffinity_numeric'] = pd.to_numeric(plot_df['pAffinity'], errors='coerce')
    plot_df = plot_df.dropna(subset=['release_year_numeric', 'pAffinity_numeric'])
    if len(plot_df) > 5000:
        plot_df = plot_df.sample(5000, random_state=42)

    palette = {'train': '#4f7fba', 'valid': '#c58b31', 'test': '#c85454'}
    fig, ax = plt.subplots(figsize=(8.2, 4.4))
    for split, part in plot_df.groupby('split'):
        ax.scatter(part['release_year_numeric'], part['pAffinity_numeric'], s=12, alpha=0.58, label=split, color=palette[split], edgecolors='none')
    ax.set_title(title)
    ax.set_xlabel('release year')
    ax.set_ylabel('pAffinity')
    ax.legend(frameon=False)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    return fig


def plot_component_assignment(comp_df, title):
    """查看 sequence component 的大小与平均 pAffinity 如何分配到三个 split。"""
    plot_df = comp_df.copy()
    plot_df['pAffinity_mean'] = pd.to_numeric(plot_df['pAffinity_mean'], errors='coerce')
    plot_df = plot_df.dropna(subset=['pAffinity_mean', 'component_size'])
    palette = {'train': '#4f7fba', 'valid': '#c58b31', 'test': '#c85454'}
    fig, ax = plt.subplots(figsize=(8.2, 4.4))
    for split, part in plot_df.groupby('assigned_split'):
        ax.scatter(part['component_size'], part['pAffinity_mean'], s=18, alpha=0.62, label=split, color=palette[split], edgecolors='none')
    ax.set_xscale('log')
    ax.set_title(title)
    ax.set_xlabel('sequence component size, log scale')
    ax.set_ylabel('component mean pAffinity')
    ax.legend(frameon=False)
    fig.tight_layout()
    display(fig)
    plt.close(fig)
    return fig


## 1. 生成两个版本

下面分别执行两种分割方法：

- `interformer_assigned`：输入集合是 Interformer 原目录中的 train/valid/test 并集，目标数量尽量保持原 Interformer 比例。
- `all_raw`：输入集合是全部 raw PDBbind rows，目标比例沿用 Interformer 的 train/valid/test 比例。

脚本会对每个 universe 重新生成 MMseqs all-vs-all 命中表，再构建 sequence components，避免复用旧命中文件造成检测口径不一致。


In [ ]:
# 版本 1：只对 Interformer 原 train/valid/test 并集重新分割。
interformer_df, interformer_components, interformer_report = create_sequence_cluster_split(
    universe='interformer_assigned',
    source_split_dir=INTERFORMER_SPLIT_DIR,
    output_split_dir=INTERFORMER_UNION_OUTPUT_SPLIT_DIR,
    output_dir=INTERFORMER_UNION_OUTPUT_DIR,
    all_vs_all_m8=ALL_VS_ALL_M8,
    min_seq_id=MIN_SEQ_ID,
    coverage=COVERAGE,
    min_length=MIN_CHAIN_LENGTH,
    seeds=OPTIMIZATION_SEEDS,
)

# 版本 2：对全部 raw PDBbind rows 重新分割。
raw_df, raw_components, raw_report = create_sequence_cluster_split(
    universe='all_raw',
    source_split_dir=INTERFORMER_SPLIT_DIR,
    output_split_dir=ALL_RAW_OUTPUT_SPLIT_DIR,
    output_dir=ALL_RAW_OUTPUT_DIR,
    all_vs_all_m8=ALL_VS_ALL_M8,
    min_seq_id=MIN_SEQ_ID,
    coverage=COVERAGE,
    min_length=MIN_CHAIN_LENGTH,
    seeds=OPTIMIZATION_SEEDS,
)

interformer_report['assignment_report'], interformer_report['validation_report'], raw_report['assignment_report'], raw_report['validation_report']


## 2. 数量比例与泄漏检查结果

如果 `cross_split_violation_count` 为 0，表示在当前 MMseqs all-vs-all 命中表和阈值下，没有发现跨 split 的 `>=40% identity` 序列相似性违规。

In [ ]:
def report_rows(name, report):
    target = report['assignment_report']['target_counts']
    assigned = report['assignment_report']['assigned_counts']
    rows = []
    for split in ['train', 'valid', 'test']:
        rows.append({
            'version': name,
            'split': split,
            'target_count': target[split],
            'assigned_count': assigned[split],
            'delta': assigned[split] - target[split],
            'assigned_ratio': assigned[split] / sum(assigned.values()),
            'leakage_detected': report['validation_report']['leakage_detected'],
            'cross_split_violation_count': report['validation_report']['cross_split_violation_count'],
        })
    return rows

comparison = pd.DataFrame(
    report_rows('interformer_assigned', interformer_report)
    + report_rows('all_raw', raw_report)
)
display(comparison)

def ratio_bar_table(df):
    view = df.copy()
    def bar(value):
        width = max(0, min(100, value * 100))
        return f"<div style='width:180px;border:1px solid #ddd;background:#f7f7f7;height:14px'><div style='width:{width:.1f}%;height:100%;background:#4f7fba'></div></div><span>{value:.3f}</span>"
    view['ratio_bar'] = view['assigned_ratio'].map(bar)
    return view[['version', 'split', 'assigned_count', 'ratio_bar', 'cross_split_violation_count']].to_html(index=False, escape=False)

display(HTML('<h3>两个版本的 split 比例</h3>'))
display(HTML(ratio_bar_table(comparison)))

## 2.1 三个类别的数据范围散点图

下面用 `release_year` 与 `pAffinity` 展示 train / valid / test 三个类别的范围。这个图不能证明 IID，但可以帮助观察新 split 是否在时间或标签分布上出现明显偏移。

In [ ]:
plot_split_range(interformer_df, 'Interformer 并集重分：train/valid/test 年份-亲和力范围')
plot_split_range(raw_df, 'All raw 重分：train/valid/test 年份-亲和力范围')


## 3. Sequence component 分配情况

下面展示每个 split 中的 sequence components 数量，以及最大的 components。大 component 会整体进入一个 split，这是避免跨 split 序列泄漏的关键。

In [ ]:
def component_summary(name, comp_df):
    return (
        comp_df.groupby('assigned_split')['component_size']
        .agg(component_count='count', complex_count='sum', max_component='max', mean_component='mean')
        .reset_index()
        .assign(version=name)
        [['version', 'assigned_split', 'component_count', 'complex_count', 'max_component', 'mean_component']]
    )

display(pd.concat([
    component_summary('interformer_assigned', interformer_components),
    component_summary('all_raw', raw_components),
], ignore_index=True))

display(HTML('<h3>Interformer 并集版本最大的 components</h3>'))
display(interformer_components.sort_values('component_size', ascending=False).head(15)[['component_id', 'assigned_split', 'component_size', 'pdb_ids']])

display(HTML('<h3>All raw 版本最大的 components</h3>'))
display(raw_components.sort_values('component_size', ascending=False).head(15)[['component_id', 'assigned_split', 'component_size', 'pdb_ids']])

## 3.1 Sequence component 分配散点图

x 轴是 sequence component 大小，y 轴是该 component 的平均 `pAffinity`，颜色表示被分配到 train / valid / test。大 component 必须整体进入同一个 split，这是保证 `<40%` 跨 split 序列相似性约束的关键。

In [ ]:
plot_component_assignment(interformer_components, 'Interformer 并集重分：component 分配')
plot_component_assignment(raw_components, 'All raw 重分：component 分配')


## 4. Interformer-compatible 输出文件

训练代码中 `BindingData.split()` 会读取 `split_folder` 下的 `timesplit_no_lig_overlap_train`、`timesplit_no_lig_overlap_val` 和 `timesplit_test`。因此下面两个目录都可以作为新的 split folder 使用。

In [ ]:
def list_output_files(folder):
    folder = Path(folder)
    return pd.DataFrame([
        {'folder': str(folder), 'file': path.name, 'line_count': sum(1 for _ in path.open())}
        for path in sorted(folder.iterdir())
        if path.is_file()
    ])

display(pd.concat([
    list_output_files(INTERFORMER_UNION_OUTPUT_SPLIT_DIR),
    list_output_files(ALL_RAW_OUTPUT_SPLIT_DIR),
], ignore_index=True))

## 5. 使用方式

如果训练配置原本通过 `split_folder` 指向 Interformer split 目录，可以把它改成下面之一：

- `split_sequence_cluster_interformer_union`
- `split_sequence_cluster_all_raw`

两者文件名保持 Interformer 读取接口一致。区别是前者只覆盖原 Interformer 主 split 并集，后者覆盖全部 raw PDBbind rows。

建议训练前再运行 02 notebook，把 `ANALYSIS_SPLIT_DIR` 设置为要使用的目录，做一次独立序列相似性检测。